In [4]:
import pandas as pd
from binance.client import Client

# Initialize the Binance client
# API keys are NOT required for public historical market data
client = Client()

symbol = "BTCUSDC"
interval = Client.KLINE_INTERVAL_30MINUTE
start_str = "1 month ago UTC"

print(f"Downloading 1 year of 30m data for {symbol}...")

# Fetch historical klines (candlestick data)
# This method handles the 1000-candle limit per request internally
klines = client.get_historical_klines(symbol, interval, start_str)

# Define column names based on Binance API response
columns = [
    'Open time', 'Open', 'High', 'Low', 'Close', 'Volume',
    'Close time', 'Quote asset volume', 'Number of trades',
    'Taker buy base asset volume', 'Taker buy quote asset volume', 'Ignore'
]

# Create DataFrame
df = pd.DataFrame(klines, columns=columns)

# Convert timestamps to readable datetime
df['Open time'] = pd.to_datetime(df['Open time'], unit='ms')

# Convert price/volume columns to numeric
numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)

# Keep only the essential columns
df = df[['Open time', 'Open', 'High', 'Low', 'Close', 'Volume']]

print(f"Successfully downloaded {len(df)} rows.")
df.to_csv("btc_usdc_30m_1year.csv", index=False)

Date range: 2025-04-23 15:19:09.943054+00:00 to 2026-04-23 15:19:09.943054+00:00
Successfully downloaded 5006 rows.
Data shape: (5006, 10)
Columns: ['open_time', 'close_time', 's', 'i', 'open', 'close', 'high', 'low', 'volume', 'n']
Date range in data: 2026-01-09 08:30:00 to 2026-04-23 15:00:00
Data saved to btc_usdc_30m_1year.csv


In [17]:
import sys
from datetime import datetime, timedelta, timezone

# Add parent directory to path to import hl module
sys.path.append('..')
import hl

# Download 1 year of 30m data for BTC/USDC from Hyperliquid
symbol = "BTC"
interval = "30m"

# Calculate date range (1 year ago to now)
end = datetime.now(timezone.utc)
start = end - timedelta(days=365)

print(f"Downloading 1 year of {interval} data for {symbol} from Hyperliquid...")
print(f"Date range: {start} to {end}")

# Calculate the number of API calls needed
# 30m interval = 48 candles per day
# 365 days = 17,520 candles total
# Each API call returns up to 5000 candles
# So we need ceil(17,520 / 5000) = 4 API calls

import math

interval_minutes = 30
candles_per_day = (24 * 60) / interval_minutes
total_days = 365
total_candles = candles_per_day * total_days
candles_per_api_call = 5000
num_api_calls = math.ceil(total_candles / candles_per_api_call)

print(f"Interval: {interval} ({interval_minutes} minutes)")
print(f"Candles per day: {candles_per_day}")
print(f"Total candles expected: {total_candles}")
print(f"Candles per API call: {candles_per_api_call}")
print(f"Number of API calls needed: {num_api_calls}")

# Calculate time window for each API call
# Each call covers 5000 candles * 30 minutes = 150,000 minutes
minutes_per_api_call = candles_per_api_call * interval_minutes
days_per_api_call = minutes_per_api_call / (24 * 60)

print(f"Days per API call: {days_per_api_call:.2f}")

# Fetch data in a loop
dfs = []
for i in range(num_api_calls):
    chunk_start = start + timedelta(days=i * days_per_api_call)
    chunk_end = min(start + timedelta(days=(i + 1) * days_per_api_call), end)
    
    print(f"\nAPI Call {i + 1}/{num_api_calls}")
    print(f"  Fetching from {chunk_start} to {chunk_end}...")
    
    try:
        chunk_df = hl.dl_ohlc_df(symbol, interval, chunk_start, chunk_end)
        if len(chunk_df) > 0:
            dfs.append(chunk_df)
            print(f"  Downloaded {len(chunk_df)} rows")
        else:
            print(f"  No data returned")
    except Exception as e:
        print(f"  Error fetching data: {e}")

# Combine all chunks
if dfs:
    df_hl = pd.concat(dfs, ignore_index=True)
    df_hl = df_hl.drop_duplicates(subset=['open_time'], keep='first')
    df_hl = df_hl.sort_values('open_time').reset_index(drop=True)
else:
    print("No data retrieved!")
    df_hl = pd.DataFrame()

print(f"\nSuccessfully downloaded {len(df_hl)} rows total.")
print(f"Data shape: {df_hl.shape}")

# Rename columns to match Binance format
df_hl = df_hl.rename(columns={
    'open_time': 'Open time',
    'open': 'Open',
    'high': 'High',
    'low': 'Low',
    'close': 'Close',
    'volume': 'Volume'
})

# Keep only the essential columns (same as Binance)
df_hl = df_hl[['Open time', 'Open', 'High', 'Low', 'Close', 'Volume']]

print(f"Date range in data: {df_hl['Open time'].min()} to {df_hl['Open time'].max()}")

# Save to CSV file
filename_hl = "btc_usdc_30m_1year_hyperliquid.csv"
df_hl.to_csv(filename_hl, index=False)
print(f"Data saved to {filename_hl}")
df_hl.head()

Date range: 2025-04-23 15:53:25.089365+00:00 to 2026-04-23 15:53:25.089365+00:00
Interval: 30m (30 minutes)
Candles per day: 48.0
Total candles expected: 17520.0
Candles per API call: 5000
Number of API calls needed: 4
Days per API call: 104.17

API Call 1/4
  Fetching from 2025-04-23 15:53:25.089365+00:00 to 2025-08-05 19:53:25.089365+00:00...
  Error fetching data: 't'

API Call 2/4
  Fetching from 2025-08-05 19:53:25.089365+00:00 to 2025-11-17 23:53:25.089365+00:00...
  Error fetching data: 't'

API Call 3/4
  Fetching from 2025-11-17 23:53:25.089365+00:00 to 2026-03-02 03:53:25.089365+00:00...
  Downloaded 2487 rows

API Call 4/4
  Fetching from 2026-03-02 03:53:25.089365+00:00 to 2026-04-23 15:53:25.089365+00:00...
  Downloaded 2521 rows

Successfully downloaded 5007 rows total.
Data shape: (5007, 10)
Date range in data: 2026-01-09 08:30:00 to 2026-04-23 15:30:00
Data saved to btc_usdc_30m_1year_hyperliquid.csv


,Open time,Open,High,Low,Close,Volume
0,2026-01-09 08:30:00,90690.0,90691.0,89645.0,90012.0,4017.79603
1,2026-01-09 09:00:00,90009.0,90491.0,89995.0,90398.0,2088.20809
2,2026-01-09 09:30:00,90399.0,90413.0,90273.0,90326.0,302.37069
3,2026-01-09 10:00:00,90327.0,90545.0,90267.0,90428.0,389.71503
4,2026-01-09 10:30:00,90428.0,90525.0,90389.0,90478.0,161.13984


In [19]:
df_hl['Open time'].min()

Timestamp('2026-01-09 08:30:00')

In [22]:
len(dfs)

2

In [34]:
import requests
import pandas as pd
import time
from datetime import datetime, timezone

# --- CONFIGURATION ---
# PASTE YOUR NEW KEY FROM dashboard.dwellir.com HERE
API_KEY = "cede3b36-6097-42a6-9e1e-5e55dd0ada53" 

# Correct Dwellir API endpoint for Hyperliquid index data
BASE_URL = f"https://api-hyperliquid-index.n.dwellir.com/ohlcv/{API_KEY}"

# FIXED: 'UBTC' is the canonical name for the BTC/USDC spot pair
SYMBOL = "UBTC" 
INTERVAL = "5m"
START_DATE = "2025-07-27T08:00:00Z" 

def fetch_dwellir_candles(symbol, interval, start_iso):
    all_candles = []
    try:
        # Standard ISO to ms timestamp conversion
        start_ts = int(datetime.fromisoformat(start_iso.replace('Z', '+00:00')).timestamp() * 1000)
    except Exception as e:
        print(f"Check your START_DATE format (YYYY-MM-DDTHH:MM:SSZ). Error: {e}")
        return None
        
    end_ts_now = int(time.time() * 1000)
    print(f"Starting fetch for {symbol}...")
    print(f"API URL: {BASE_URL}")
    print(f"Start timestamp: {start_ts}, End timestamp: {end_ts_now}")

    current_start = start_ts
    call_count = 0
    while current_start < end_ts_now:
        call_count += 1
        if call_count > 10:  # Safety limit to prevent infinite loops during testing
            print("Reached call limit (10) - stopping")
            break
            
        payload = {
            "method": "get_ohlcv",
            "params": {
                "symbol": symbol,
                "interval": interval,
                "startTime": current_start,
                "limit": 1000 
            }
        }
        
        print(f"\nCall {call_count}: Sending payload: {payload}")
        
        try:
            # Dwellir requires a POST request for OHLCV
            response = requests.post(BASE_URL, json=payload, timeout=20)
            print(f"Response status: {response.status_code}")
            print(f"Response text: {response.text[:200]}")
            
            if response.status_code != 200:
                print(f"Error {response.status_code}: {response.text[:200]}")
                break
                
            data = response.json()
            print(f"Parsed response: {str(data)[:200]}")
            
            if "result" in data and data["result"]:
                batch = data["result"]
                all_candles.extend(batch)
                print(f"Got {len(batch)} new candles, total: {len(all_candles)}")
                
                # Update timestamp for next loop (Batch format: [timestamp, o, h, l, c, v])
                last_candle_ts = batch[-1][0] 
                
                if last_candle_ts <= current_start:
                    print("No new data in this batch - stopping")
                    break
                
                current_start = last_candle_ts + 1
                print(f"Fetched until: {datetime.fromtimestamp(last_candle_ts/1000, tz=timezone.utc)}")
                time.sleep(0.2)  # Rate limit safety
            else:
                print("No result in response or empty result")
                break
        except Exception as e:
            print(f"Exception error: {e}")
            import traceback
            traceback.print_exc()
            break

    if not all_candles:
        print("No candles collected")
        return None

    # Reconstruct the OHLCV dataframe
    df = pd.DataFrame(all_candles, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms')
    return df.drop_duplicates(subset=['timestamp']).sort_values('timestamp')

# Run the fetcher
df_final = fetch_dwellir_candles(SYMBOL, INTERVAL, START_DATE)

if df_final is not None:
    df_final.to_csv("hl_ubtc_5m.csv", index=False)
    print(f"Success! Saved {len(df_final)} rows to hl_ubtc_5m.csv")
else:
    print("Download failed. Verify your API Key at https://dashboard.dwellir.com/")

Starting fetch for UBTC...
API URL: https://api-hyperliquid-index.n.dwellir.com/ohlcv/cede3b36-6097-42a6-9e1e-5e55dd0ada53
Start timestamp: 1753603200000, End timestamp: 1777185850777

Call 1: Sending payload: {'method': 'get_ohlcv', 'params': {'symbol': 'UBTC', 'interval': '5m', 'startTime': 1753603200000, 'limit': 1000}}
Response status: 403
Response text: API key does not exist. Please contact Dwellir at support@dwellir.com if this is unexpected.

Error 403: API key does not exist. Please contact Dwellir at support@dwellir.com if this is unexpected.

No candles collected
Download failed. Verify your API Key at https://dashboard.dwellir.com/
